# Step 2: Choosing learners

## Imports

In [1]:
import warnings

import numpy as np
import pandas as pd
from IPython.display import display
from scipy.stats import f_oneway
from sklearn.compose import ColumnTransformer
from sklearn.ensemble import GradientBoostingClassifier, RandomForestClassifier
from sklearn.impute import SimpleImputer
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import StratifiedKFold, cross_validate, train_test_split
from sklearn.naive_bayes import GaussianNB
from sklearn.neighbors import KNeighborsClassifier
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.svm import SVC
from sklearn.tree import DecisionTreeClassifier

warnings.filterwarnings("ignore", category=FutureWarning)
np.random.seed(42)
RANDOM_STATE = 42
CV_SPLITS = 10
LASTNAME = "cheng"


## Rebuild Step 1 data (load → targets → split → same column order as spec)

Paths are relative.

In [2]:
SENTINEL = 5.397605346934028e-79

nhanes = pd.read_csv("../datasets/nhanes.csv")
nhanes_clean = nhanes.replace(SENTINEL, np.nan)


def compute_bp(row):
    sy, di = row["BPXOSY1"], row["BPXODI1"]
    if pd.isna(sy) or pd.isna(di):
        return np.nan
    return "normal" if (sy < 120 and di < 80) else "abnormal"


def compute_chol(val):
    if pd.isna(val):
        return np.nan
    if val < 200:
        return "normal"
    if val <= 239:
        return "borderline"
    return "high"


nhanes_clean["BP"] = nhanes_clean.apply(compute_bp, axis=1)
nhanes_clean["CHOL"] = nhanes_clean["LBXTC"].apply(compute_chol)
nhanes_clean = nhanes_clean.drop(columns=["BPXOSY1", "BPXODI1", "LBXTC"])

REQUIRED_COLS = [
    "SEQN", "SDDSRVYR", "RIDSTATR", "RIAGENDR", "RIDAGEYR", "RIDRETH1",
    "DMQMILIZ", "DMDBORN4", "DMDYRUSR", "DMDEDUC2", "DMDMARTZ", "DMDHHSIZ",
    "WTINT2YR", "WTMEC2YR", "ALQ111", "ALQ121", "ALQ130", "ALQ151", "ALQ170",
    "HIQ011", "HIQ032A", "HIQ032B", "HIQ032D", "HIQ032F", "HIQ032H", "HIQ032I",
    "HIQ210", "INDFMMPC", "INQ300", "IND310", "BP", "CHOL",
]
nhanes_clean = nhanes_clean[REQUIRED_COLS]

bp_df = nhanes_clean.copy()
chol_df = nhanes_clean.copy()

bp_training_cheng, bp_test_cheng = train_test_split(
    bp_df, test_size=0.2, random_state=42
)
chol_training_cheng, chol_test_cheng = train_test_split(
    chol_df, test_size=0.2, random_state=42
)

FEATURE_COLS = [c for c in REQUIRED_COLS if c not in ["SEQN", "BP", "CHOL"]]
NUMERIC_COLS = ["RIDAGEYR", "WTINT2YR", "WTMEC2YR", "DMDHHSIZ"]
CATEGORICAL_COLS = [c for c in FEATURE_COLS if c not in NUMERIC_COLS]


def make_preprocessing_pipeline():
    numeric_transformer = Pipeline(
        [
            ("impute", SimpleImputer(strategy="median")),
            ("scale", StandardScaler()),
        ]
    )
    categorical_transformer = Pipeline(
        [
            ("impute", SimpleImputer(strategy="most_frequent")),
            (
                "encode",
                OneHotEncoder(handle_unknown="ignore", sparse_output=False),
            ),
        ]
    )
    return ColumnTransformer(
        [
            ("num", numeric_transformer, NUMERIC_COLS),
            ("cat", categorical_transformer, CATEGORICAL_COLS),
        ]
    )


def get_X_y(train_df, target_col):
    """Feature matrix and labels; drop rows with missing target."""
    X = train_df[FEATURE_COLS].replace(SENTINEL, np.nan)
    y = train_df[target_col]
    mask = y.notna()
    return X.loc[mask], y.loc[mask]


X_bp, y_bp = get_X_y(bp_training_cheng, "BP")
X_chol, y_chol = get_X_y(chol_training_cheng, "CHOL")

print("BP training rows (with label):", len(y_bp), y_bp.value_counts().to_dict())
print("CHOL training rows (with label):", len(y_chol), y_chol.value_counts().to_dict())


BP training rows (with label): 4407 {'abnormal': 2502, 'normal': 1905}
CHOL training rows (with label): 4115 {'normal': 2657, 'borderline': 1018, 'high': 440}


## Task 6

| # | Assignment bucket | Estimator |
|---|-------------------|-----------|
| 1 | Decision stump | `DecisionTreeClassifier(max_depth=1)` |
| 2 | Larger tree | `DecisionTreeClassifier(max_depth=20, ...)` |
| 3 | Ensemble | `RandomForestClassifier` |
| 4 | Ensemble | `GradientBoostingClassifier` |
| 5 | SVM | `SVC(kernel='linear')` |
| 6 | Bayesian | `GaussianNB` |
| 7 | Other (textbook) | `KNeighborsClassifier` |
| 8 | Other (textbook) | `LogisticRegression` |

Each model is wrapped in `Pipeline([preprocess, clf])` so each CV fold fits preprocessing only on training folds (no leakage).

In [3]:
def build_estimators():
    return {
        "1_decision_stump": DecisionTreeClassifier(max_depth=1, random_state=RANDOM_STATE),
        "2_decision_tree": DecisionTreeClassifier(
            max_depth=20, min_samples_leaf=10, random_state=RANDOM_STATE
        ),
        "3_random_forest": RandomForestClassifier(
            n_estimators=100, random_state=RANDOM_STATE, n_jobs=-1
        ),
        "4_gradient_boosting": GradientBoostingClassifier(random_state=RANDOM_STATE),
        "5_linear_svm": SVC(kernel="linear", random_state=RANDOM_STATE),
        "6_gaussian_nb": GaussianNB(),
        "7_knn": KNeighborsClassifier(n_neighbors=15),
        "8_logistic_regression": LogisticRegression(
            max_iter=5000, random_state=RANDOM_STATE
        ),
    }


ESTIMATORS = build_estimators()
SCORING = ["accuracy", "f1_macro", "balanced_accuracy"]
ANOVA_METRIC = "f1_macro"

cv = StratifiedKFold(
    n_splits=CV_SPLITS, shuffle=True, random_state=RANDOM_STATE
)


def run_task6(X, y, task_name):
    rows = []
    fold_scores_for_anova = {}
    for name, est in ESTIMATORS.items():
        pipe = Pipeline([("pre", make_preprocessing_pipeline()), ("clf", est)])
        out = cross_validate(
            pipe,
            X,
            y,
            cv=cv,
            scoring=SCORING,
            n_jobs=-1,
        )
        row = {"algorithm": name, "task": task_name}
        for m in SCORING:
            key = "test_" + m
            scores = out[key]
            row[m + "_mean"] = float(np.mean(scores))
            row[m + "_std"] = float(np.std(scores))
        rows.append(row)
        fold_scores_for_anova[name] = out["test_" + ANOVA_METRIC]
    return pd.DataFrame(rows), fold_scores_for_anova


results_bp, folds_bp = run_task6(X_bp, y_bp, "blood_pressure")
results_chol, folds_chol = run_task6(X_chol, y_chol, "cholesterol")

display(results_bp.round(4))
display(results_chol.round(4))


,algorithm,task,accuracy_mean,accuracy_std,f1_macro_mean,f1_macro_std,balanced_accuracy_mean,balanced_accuracy_std
0,1_decision_stump,blood_pressure,0.6555,0.0307,0.6390,0.0332,0.6386,0.0323
1,2_decision_tree,blood_pressure,0.6072,0.0279,0.5939,0.0295,0.5937,0.0290
2,3_random_forest,blood_pressure,0.6417,0.0150,0.6188,0.0196,0.6202,0.0179
3,4_gradient_boosting,blood_pressure,0.6678,0.0317,0.6416,0.0365,0.6432,0.0339
4,5_linear_svm,blood_pressure,0.6594,0.0305,0.6410,0.0330,0.6406,0.0318
5,6_gaussian_nb,blood_pressure,0.5364,0.0641,0.4558,0.0784,0.5304,0.0263
6,7_knn,blood_pressure,0.6329,0.0195,0.6120,0.0212,0.6128,0.0203
7,8_logistic_regression,blood_pressure,0.6607,0.0325,0.6436,0.0344,0.6429,0.0333


,algorithm,task,accuracy_mean,accuracy_std,f1_macro_mean,f1_macro_std,balanced_accuracy_mean,balanced_accuracy_std
0,1_decision_stump,cholesterol,0.6457,0.0008,0.2616,0.0002,0.3333,0.0000
1,2_decision_tree,cholesterol,0.5577,0.0226,0.3776,0.0299,0.3756,0.0272
2,3_random_forest,cholesterol,0.6343,0.0100,0.3205,0.0187,0.3527,0.0114
3,4_gradient_boosting,cholesterol,0.6408,0.0095,0.2911,0.0188,0.3423,0.0107
4,5_linear_svm,cholesterol,0.6435,0.0022,0.2631,0.0032,0.3328,0.0017
5,6_gaussian_nb,cholesterol,0.1230,0.0053,0.0968,0.0089,0.3381,0.0032
6,7_knn,cholesterol,0.6294,0.0105,0.3283,0.0200,0.3556,0.0122
7,8_logistic_regression,cholesterol,0.6382,0.0078,0.2733,0.0119,0.3344,0.0075


### One-way ANOVA (95%)

In [4]:
groups_bp = [folds_bp[k] for k in sorted(folds_bp)]
f_bp, p_bp = f_oneway(*groups_bp)
groups_chol = [folds_chol[k] for k in sorted(folds_chol)]
f_chol, p_chol = f_oneway(*groups_chol)

anova_rows = [
    {
        "task": "blood_pressure (BP)",
        "metric_for_anova": "macro F1 (test_" + ANOVA_METRIC + " per fold)",
        "F_statistic": f_bp,
        "p_value": p_bp,
        "reject_H0_at_alpha_0.05": p_bp < 0.05,
    },
    {
        "task": "cholesterol (CHOL)",
        "metric_for_anova": "macro F1 (test_" + ANOVA_METRIC + " per fold)",
        "F_statistic": f_chol,
        "p_value": p_chol,
        "reject_H0_at_alpha_0.05": p_chol < 0.05,
    },
]
anova_df = pd.DataFrame(anova_rows)
display(anova_df)

print(f"F = {f_bp:.6g}, p = {p_bp:.6g}")
print(f"F = {f_chol:.6g}, p = {p_chol:.6g}")


,task,metric_for_anova,F_statistic,p_value,reject_H0_at_alpha_0.05
0,blood_pressure (BP),macro F1 (test_f1_macro per fold),22.861006,5.432521e-16,True
1,cholesterol (CHOL),macro F1 (test_f1_macro per fold),220.071345,5.838641e-46,True


F = 22.861, p = 5.43252e-16
F = 220.071, p = 5.83864e-46


## Task 7

In [5]:
def top_three(df):
    return df.sort_values("f1_macro_mean", ascending=False).head(3)[["algorithm", "f1_macro_mean", "accuracy_mean", "balanced_accuracy_mean"]]

print("Top 3 by mean macro F1 — BP")
display(top_three(results_bp))
print("Top 3 by mean macro F1 — CHOL")
display(top_three(results_chol))


Top 3 by mean macro F1 — BP


,algorithm,f1_macro_mean,accuracy_mean,balanced_accuracy_mean
7,8_logistic_regression,0.643560,0.660740,0.642889
3,4_gradient_boosting,0.641550,0.667779,0.643202
4,5_linear_svm,0.640967,0.659385,0.640630


Top 3 by mean macro F1 — CHOL


,algorithm,f1_macro_mean,accuracy_mean,balanced_accuracy_mean
1,2_decision_tree,0.377579,0.557714,0.375563
6,7_knn,0.328252,0.629401,0.355606
2,3_random_forest,0.320548,0.634266,0.352744
